In [8]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error


for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [9]:
data_dir = '/kaggle/input/competitions/house-prices-advanced-regression-techniques'

train = pd.read_csv(f'{data_dir}/train.csv')
test = pd.read_csv(f'{data_dir}/test.csv')
sample_submission = pd.read_csv(f'{data_dir}/sample_submission.csv')

print("Train", train.shape)
print('Test', test.shape)
print("Sample Submission", sample_submission.shape)

display(train.head())


missing = train.isna().sum()
missing = missing[missing>0].sort_values(ascending=False)
display(missing)
print('Training duplicate rows:', train.duplicated().sum())

Train (1460, 81)
Test (1459, 80)
Sample Submission (1459, 2)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

Training duplicate rows: 0


#Dropping rows and verifying number of columns with categorical variables
target='SalePrice'
X=train.drop(columns=[target].copy())
Y=train[target].copy()
X_test=test.copy()

numeric_columns = X.select_dtypes(include ='number').columns.tolist()
categorical_columns = X.select_dtypes(exclude = 'number').columns.tolist()

print("Numerical predictor columns: =", len(numeric_columns))
print("Categorical predictor columns: =", len(categorical_columns))
print("\nCategorical columns:")
print(categorical_columns)


In [10]:
target = "SalePrice"
X = train.drop(columns=[target]).copy()
y = train[target].copy()
X_test = test.copy()

numeric_columns = X.select_dtypes(include="number").columns.tolist()
categorical_columns = X.select_dtypes(exclude="number").columns.tolist()

print("Numerical predictor columns:", len(numeric_columns))
print("Categorical predictor columns:", len(categorical_columns))

print("\nCategorical columns:")
print(categorical_columns)
missing_columns = X.columns[X.isna().any()].tolist()
print('Columns with missing values:')
for column in missing_columns:
    print(f"{column:15} missing={X[column].isna().sum():4} d_type={X[column].dtype}")

Numerical predictor columns: 37
Categorical predictor columns: 43

Categorical columns:
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']
Columns with missing values:
LotFrontage     missing= 259 d_type=float64
Alley           missing=1369 d_type=object
MasVnrType      missing= 872 d_type=object
MasVnrArea      missing=   8 d_type=float64
BsmtQual        missing=  37 d_type=object
BsmtCond        missing=  37 d_type=object
BsmtExposure    missing=  38 d_type=object


In [11]:
# Columns where a missing value means the feature is absent
categorical_none = [
    "Alley", "MasVnrType", "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2", "FireplaceQu", "GarageType",
    "GarageFinish", "GarageQual", "GarageCond", "PoolQC", "Fence",
    "MiscFeature",
]

numerical_none = ["MasVnrArea", "GarageYrBlt"]

# Work on copies
X_clean = X.copy()
X_test_clean = X_test.copy()

for column in categorical_none:
    X_clean[column] = X_clean[column].fillna('None')
    X_test_clean[column] = X_test_clean[column].fillna('None')

for column in numerical_none:
    X_clean[column] = X_clean[column].fillna(0)
    X_test_clean[column] = X_test_clean[column].fillna(0)

In [12]:
# for lot frontage use median
for data in [X_clean, X_test_clean]:
    data["LotFrontage"] = data.groupby('Neighborhood')["LotFrontage"].transform(lambda values: values.fillna(values.median()))
    data["LotFrontage"] = data['LotFrontage'].fillna(X_clean["LotFrontage"].median())

# for remaining training median
for column in X_clean.columns:
    if X_clean[column].isna().any() or X_test_clean[column].isna().any():
        if X_clean[column].dtype=="object":
            most_common = X_clean[column].mode()[0]
            X_clean[column] = X_clean[column].fillna(most_common)
            X_test_clean[column] = X_test_clean[column].fillna(most_common)
        else:
            median_value = X_clean[column].median()
            X_clean[column] = X_clean[column].fillna(median_value)
            X_test_clean[column] = X_test_clean[column].fillna(median_value)

print("Missing values in cleaned training predictors:", X_clean.isna().sum().sum())
print("Missing values in cleaned test predictors:", X_test_clean.isna().sum().sum())
    

Missing values in cleaned training predictors: 0
Missing values in cleaned test predictors: 0


In [13]:
all_data = pd.concat([X_clean, X_test_clean], keys = ["train", "test"])
all_data_encoded = pd.get_dummies(all_data, dtype=int)
X_encoded = all_data_encoded.xs("train")
X_test_encoded = all_data_encoded.xs("test")
print("Encoded training shape:",X_encoded.shape)
print("Encoded test shape:",X_test_encoded.shape)
print("Same columns:", X_encoded.columns.equals(X_test_encoded.columns))
display(X_encoded.head())



Encoded training shape: (1460, 303)
Encoded test shape: (1459, 303)
Same columns: True


,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,60,65.0,8450,7,5,2003,2003,196.0,706.0,...,0,0,0,1,0,0,0,0,1,0
1,2,20,80.0,9600,6,8,1976,1976,0.0,978.0,...,0,0,0,1,0,0,0,0,1,0
2,3,60,68.0,11250,7,5,2001,2002,162.0,486.0,...,0,0,0,1,0,0,0,0,1,0
3,4,70,60.0,9550,7,5,1915,1970,0.0,216.0,...,0,0,0,1,1,0,0,0,0,0
4,5,60,84.0,14260,8,5,2000,2000,350.0,655.0,...,0,0,0,1,0,0,0,0,1,0


In [14]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# Copy the cleaned predictors
X_cat_subclass = X_clean.copy()
X_test_cat_subclass = X_test_clean.copy()

# Treat text columns and MSSubClass as categorical
categorical_columns = X_cat_subclass.select_dtypes(
    include=["object", "category"]
).columns.tolist()

if "MSSubClass" not in categorical_columns:
    categorical_columns.append("MSSubClass")

# CatBoost expects categorical values to be strings
for column in categorical_columns:
    X_cat_subclass[column] = X_cat_subclass[column].astype(str)
    X_test_cat_subclass[column] = X_test_cat_subclass[column].astype(str)

log_target = np.log1p(y)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
oof_predictions = np.zeros(len(y))
best_iterations = []

# Cross-validate
for train_idx, valid_idx in cv.split(X_cat_subclass):
    model = CatBoostRegressor(
        iterations=3000,
        depth=4,
        learning_rate=0.03,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )

    model.fit(
        X_cat_subclass.iloc[train_idx],
        log_target.iloc[train_idx],
        cat_features=categorical_columns,
        eval_set=(
            X_cat_subclass.iloc[valid_idx],
            log_target.iloc[valid_idx],
        ),
        early_stopping_rounds=150,
        verbose=False,
    )

    oof_predictions[valid_idx] = model.predict(
        X_cat_subclass.iloc[valid_idx]
    )
    best_iterations.append(model.get_best_iteration() + 1)

cv_rmsle = np.sqrt(mean_squared_error(log_target, oof_predictions))
final_iterations = int(np.median(best_iterations))

print("CatBoost CV RMSLE:", round(cv_rmsle, 5))
print("Best iterations by fold:", best_iterations)
print("Final iterations:", final_iterations)

# Fit on all training rows
final_model_msclass = CatBoostRegressor(
    iterations=final_iterations,
    depth=4,
    learning_rate=0.03,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

final_model_msclass.fit(
    X_cat_subclass,
    log_target,
    cat_features=categorical_columns,
    verbose=False,
)

# Predict test prices and convert from log prices
predictions = np.maximum(
    np.expm1(final_model_msclass.predict(X_test_cat_subclass)),
    0,
)

# Create and save the submission
submission_msclass = sample_submission.copy()
submission_msclass["SalePrice"] = predictions
submission_msclass.to_csv(
    "/kaggle/working/submission_catboost_msclass.csv",
    index=False,
)

print(submission_msclass.head())
print("Saved submission_catboost_msclass.csv")


CatBoost CV RMSLE: 0.12462
Best iterations by fold: [1585, 2202, 174, 3000, 2965]
Final iterations: 2202
     Id      SalePrice
0  1461  118811.274241
1  1462  157867.577473
2  1463  190026.652750
3  1464  195334.555831
4  1465  183716.890338
Saved submission_catboost_msclass.csv


In [17]:
import os
import os

file_path = "/kaggle/working/submission_catboost_msclass.csv"
print(os.path.exists(file_path))
print("Files in working folder:", os.listdir("/kaggle/working"))

True
Files in working folder: ['submission_catboost_msclass.csv', '.virtual_documents']


In [18]:
from IPython.display import FileLink, display

display(FileLink("submission_catboost_msclass.csv"))

/kaggle/working/submission_catboost_msclass.csv